# GRC Pro: contextual risk of Low-criticality asset combinations
Pipeline: select 2-4 Low assets -> features -> CatBoost -> SHAP and LIME -> candidate ISO controls -> Maximum Coverage action plan.

## 1. Data, features and split

In [1]:
import random
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("pandas version:", pd.__version__)
print("numpy version :", np.__version__)
print("Random seed   :", SEED)

pandas version: 3.0.1
numpy version : 2.4.3
Random seed   : 42


In [2]:
import os

CODE_DIR = r"C:\Users\Vidya Bodasakurthi\Downloads\GRC MINOR\Claude"
PATH = r"C:\Users\Vidya Bodasakurthi\Downloads\GRC MINOR\Datasets\ERG.xlsx"

os.chdir(CODE_DIR)

print("Working folder:", os.getcwd())
print("ERG.xlsx found:", os.path.exists(PATH))

Working folder: C:\Users\Vidya Bodasakurthi\Downloads\GRC MINOR\Claude
ERG.xlsx found: True


In [3]:
from data_loader import EnterpriseInventory
from feature_extraction import compute_features
from model_utils import prepare_inference_row
from explain import shap_explain, lime_explain
from report import build_report

print("All project modules imported successfully")

All project modules imported successfully


In [4]:
inv = EnterpriseInventory(PATH)

assets_df = inv.assets_df
risk_df = inv.risk_df

print("Total assets:", len(assets_df))
print("Asset columns:", list(assets_df.columns))

[EnterpriseInventory] Loaded 1082 assets, 649 Low-critical, from C:\Users\Vidya Bodasakurthi\Downloads\GRC MINOR\Datasets\ERG.xlsx
Total assets: 1082
Asset columns: ['asset_id', 'asset_name', 'asset_type', 'asset_subtype', 'technology_platform', 'department', 'asset_owner', 'business_process_id', 'environment', 'deployment_type', 'location', 'internet_facing', 'lifecycle_status', 'asset_age_years', 'vendor', 'number_of_users']


In [5]:
print(risk_df["existing_criticality"].value_counts())

low_pool_df = risk_df[risk_df["existing_criticality"] == "Low"]
LOW_ASSET_POOL_IDS = set(low_pool_df["asset_id"])

print("\nLow-criticality assets in the pool:", len(LOW_ASSET_POOL_IDS))

existing_criticality
Low       649
Medium    292
High      141
Name: count, dtype: int64

Low-criticality assets in the pool: 649


In [6]:
features_df = pd.read_excel(PATH, sheet_name="05_Combination_Features")
labels_df = pd.read_excel(PATH, sheet_name="06_Combined_Criticality_Labels")

print("Features table:", features_df.shape)
print("Labels table  :", labels_df.shape)
print()
print(labels_df["combined_criticality"].value_counts())

Features table: (14000, 61)
Labels table  : (14000, 2)

combined_criticality
Low       5705
Medium    5507
High      2788
Name: count, dtype: int64


In [7]:
splits_df = pd.read_excel(PATH, sheet_name="11_Model_TrainTest_Split")

check_df = labels_df.merge(splits_df, on="combination_id")

print(splits_df["split"].value_counts())
print()
print(pd.crosstab(check_df["split"], check_df["combined_criticality"]))

split
train         9799
validation    2101
test          2100
Name: count, dtype: int64

combined_criticality  High   Low  Medium
split                                   
test                   418   856     826
train                 1952  3993    3854
validation             418   856     827


In [8]:
df = features_df.merge(labels_df, on="combination_id").merge(splits_df, on="combination_id")

not_features = ["combination_id", "combined_criticality", "split"]
raw_feature_cols = [c for c in df.columns if c not in not_features]

cat_cols = [c for c in df.select_dtypes(include="object").columns if c not in not_features]

print("Combined table shape:", df.shape)
print("Raw contextual features:", len(raw_feature_cols))
print("Text (categorical) columns:", cat_cols)

Combined table shape: (14000, 63)
Raw contextual features: 60
Text (categorical) columns: ['combined_information_sensitivity', 'confidentiality_requirement', 'integrity_requirement', 'availability_requirement', 'business_process_importance']


C:\Users\Vidya Bodasakurthi\AppData\Local\Temp\ipykernel_10780\2130064257.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = [c for c in df.select_dtypes(include="object").columns if c not in not_features]


In [9]:
bool_cols = df.select_dtypes(include="bool").columns.tolist()

df_enc = pd.get_dummies(df, columns=cat_cols, drop_first=False)

feature_cols = [c for c in df_enc.columns if c not in not_features]

new_columns = [c for c in feature_cols if c not in raw_feature_cols]

print("Model input columns:", len(feature_cols))
print("New yes/no columns created:", len(new_columns))
print("Boolean columns in raw data:", bool_cols)
print()
print(new_columns)

Model input columns: 71
New yes/no columns created: 16
Boolean columns in raw data: []

['combined_information_sensitivity_High', 'combined_information_sensitivity_Low', 'combined_information_sensitivity_Medium', 'confidentiality_requirement_High', 'confidentiality_requirement_Low', 'confidentiality_requirement_Medium', 'integrity_requirement_High', 'integrity_requirement_Low', 'integrity_requirement_Medium', 'availability_requirement_High', 'availability_requirement_Low', 'availability_requirement_Medium', 'business_process_importance_critical', 'business_process_importance_mixed', 'business_process_importance_moderate', 'business_process_importance_significant']


In [10]:
low_pool_df = assets_df[assets_df["asset_id"].isin(LOW_ASSET_POOL_IDS)]
low_pool_df = low_pool_df[["asset_id", "asset_name", "asset_type"]].sort_values("asset_id")

print("Low-criticality assets available for selection:", len(low_pool_df))
print(low_pool_df.to_string(index=False))

Low-criticality assets available for selection: 649
asset_id                       asset_name             asset_type
 AST0001                       Laptop-001            Workstation
 AST0002                       Laptop-002            Workstation
 AST0005                       Laptop-005            Workstation
 AST0009                       Laptop-009            Workstation
 AST0010                       Laptop-010            Workstation
 AST0011                       Laptop-011            Workstation
 AST0012                       Laptop-012            Workstation
 AST0013                       Laptop-013            Workstation
 AST0014                       Laptop-014            Workstation
 AST0015                       Laptop-015            Workstation
 AST0017                       Laptop-017            Workstation
 AST0019                       Laptop-019            Workstation
 AST0020                       Laptop-020            Workstation
 AST0022                       Laptop-

## 2. Select assets and generate features

In [11]:
raw_input_text = input("Enter 2-4 Asset IDs from the Low pool, separated by commas: ")
selected_assets = [a.strip().upper() for a in raw_input_text.split(",") if a.strip()]

problems = []
if len(selected_assets) < 2 or len(selected_assets) > 4:
    problems.append("You must select between 2 and 4 assets")
if len(set(selected_assets)) != len(selected_assets):
    problems.append("The same asset was entered twice")
for a in selected_assets:
    if a not in LOW_ASSET_POOL_IDS:
        problems.append(a + " is not in the Low-criticality pool")

if len(problems) > 0:
    print("Selection is not valid:")
    for p in problems:
        print(" -", p)
else:
    print("Selection is valid:", selected_assets)
    print(assets_df[assets_df["asset_id"].isin(selected_assets)][["asset_id", "asset_name", "asset_type"]].to_string(index=False))

Enter 2-4 Asset IDs from the Low pool, separated by commas:  AST1066,AST1079,AST1082


Selection is valid: ['AST1066', 'AST1079', 'AST1082']
asset_id                    asset_name             asset_type
 AST1066 ApplicationWorkerInstance-074 Cloud Compute Instance
 AST1079 ApplicationWorkerInstance-087 Cloud Compute Instance
 AST1082 ApplicationWorkerInstance-090 Cloud Compute Instance


In [12]:
raw_features = compute_features(selected_assets, inv)

print("Number of features generated:", len(raw_features))
print("Combination size feature    :", raw_features["combination_size"])
print()
for name in ["asset_type_diversity", "combined_information_sensitivity", "relationship_count",
             "shared_user_count", "common_vulnerability_count", "known_exploited_present",
             "control_gap_count"]:
    print(name, "=", raw_features[name])

Number of features generated: 60
Combination size feature    : 3

asset_type_diversity = 1
combined_information_sensitivity = Low
relationship_count = 0
shared_user_count = 0
common_vulnerability_count = 0
known_exploited_present = False
control_gap_count = 3


## 3. Models (five, compared on validation macro F1)

In [13]:
from sklearn.preprocessing import LabelEncoder

train_df = df_enc[df_enc["split"] == "train"]
val_df = df_enc[df_enc["split"] == "validation"]
test_df = df_enc[df_enc["split"] == "test"]

le = LabelEncoder()
y_train = le.fit_transform(train_df["combined_criticality"])
y_val = le.transform(val_df["combined_criticality"])
y_test = le.transform(test_df["combined_criticality"])

X_train = train_df[feature_cols]
X_val = val_df[feature_cols]
X_test = test_df[feature_cols]

print("Train:", X_train.shape, "| Validation:", X_val.shape, "| Test:", X_test.shape)
print("Class order:", list(le.classes_))

Train: (9799, 71) | Validation: (2101, 71) | Test: (2100, 71)
Class order: ['High', 'Low', 'Medium']


In [14]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

model_results = {}

def evaluate(name, model):
    val_pred = model.predict(X_val)
    test_pred = model.predict(X_test)
    model_results[name] = {
        "val_accuracy": accuracy_score(y_val, val_pred),
        "val_precision": precision_score(y_val, val_pred, average="macro"),
        "val_recall": recall_score(y_val, val_pred, average="macro"),
        "val_f1": f1_score(y_val, val_pred, average="macro"),
        "test_accuracy": accuracy_score(y_test, test_pred),
        "test_precision": precision_score(y_test, test_pred, average="macro"),
        "test_recall": recall_score(y_test, test_pred, average="macro"),
        "test_f1": f1_score(y_test, test_pred, average="macro"),
    }
    table = pd.DataFrame(model_results).T[["val_accuracy", "val_precision", "val_recall", "val_f1"]]
    table.columns = ["Accuracy", "Precision", "Recall", "F1"]
    print("VALIDATION RESULTS so far (macro-averaged)")
    print(table.round(4).to_string())

print("Evaluation helper ready")

Evaluation helper ready


In [15]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

model_results = {}
metric_names = ["Accuracy", "Precision (macro)", "Recall (macro)", "F1 (macro)"]

nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

val_pred = nb_model.predict(X_val)
test_pred = nb_model.predict(X_test)

val_scores = [accuracy_score(y_val, val_pred),
              precision_score(y_val, val_pred, average="macro"),
              recall_score(y_val, val_pred, average="macro"),
              f1_score(y_val, val_pred, average="macro")]
test_scores = [accuracy_score(y_test, test_pred),
               precision_score(y_test, test_pred, average="macro"),
               recall_score(y_test, test_pred, average="macro"),
               f1_score(y_test, test_pred, average="macro")]

model_results["Naive Bayes"] = {"validation": val_scores, "test": test_scores}

print("GAUSSIAN NAIVE BAYES")
print(pd.DataFrame({"Validation": val_scores, "Test": test_scores}, index=metric_names).round(4))

GAUSSIAN NAIVE BAYES
                   Validation    Test
Accuracy               0.7425  0.7357
Precision (macro)      0.7543  0.7487
Recall (macro)         0.6728  0.6575
F1 (macro)             0.6806  0.6596


In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

rf_model = RandomForestClassifier(n_estimators=300, random_state=42)
rf_model.fit(X_train, y_train)

val_pred = rf_model.predict(X_val)
test_pred = rf_model.predict(X_test)

val_scores = [accuracy_score(y_val, val_pred),
              precision_score(y_val, val_pred, average="macro"),
              recall_score(y_val, val_pred, average="macro"),
              f1_score(y_val, val_pred, average="macro")]
test_scores = [accuracy_score(y_test, test_pred),
               precision_score(y_test, test_pred, average="macro"),
               recall_score(y_test, test_pred, average="macro"),
               f1_score(y_test, test_pred, average="macro")]

model_results["Random Forest"] = {"validation": val_scores, "test": test_scores}

print("RANDOM FOREST")
print(pd.DataFrame({"Validation": val_scores, "Test": test_scores}, index=metric_names).round(4))

RANDOM FOREST
                   Validation    Test
Accuracy               0.9524  0.9486
Precision (macro)      0.9544  0.9485
Recall (macro)         0.9445  0.9408
F1 (macro)             0.9491  0.9444


In [17]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

xgb_model = XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.08,
                          random_state=42, eval_metric="mlogloss")
xgb_model.fit(X_train, y_train)

val_pred = xgb_model.predict(X_val)
test_pred = xgb_model.predict(X_test)

val_scores = [accuracy_score(y_val, val_pred),
              precision_score(y_val, val_pred, average="macro"),
              recall_score(y_val, val_pred, average="macro"),
              f1_score(y_val, val_pred, average="macro")]
test_scores = [accuracy_score(y_test, test_pred),
               precision_score(y_test, test_pred, average="macro"),
               recall_score(y_test, test_pred, average="macro"),
               f1_score(y_test, test_pred, average="macro")]

model_results["XGBoost"] = {"validation": val_scores, "test": test_scores}

print("XGBOOST")
print(pd.DataFrame({"Validation": val_scores, "Test": test_scores}, index=metric_names).round(4))

XGBOOST
                   Validation    Test
Accuracy               0.9657  0.9681
Precision (macro)      0.9664  0.9685
Recall (macro)         0.9621  0.9648
F1 (macro)             0.9642  0.9665


In [18]:
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

lgbm_model = LGBMClassifier(n_estimators=400, max_depth=6, learning_rate=0.08,
                            random_state=42, verbose=-1)
lgbm_model.fit(X_train, y_train)

val_pred = lgbm_model.predict(X_val)
test_pred = lgbm_model.predict(X_test)

val_scores = [accuracy_score(y_val, val_pred),
              precision_score(y_val, val_pred, average="macro"),
              recall_score(y_val, val_pred, average="macro"),
              f1_score(y_val, val_pred, average="macro")]
test_scores = [accuracy_score(y_test, test_pred),
               precision_score(y_test, test_pred, average="macro"),
               recall_score(y_test, test_pred, average="macro"),
               f1_score(y_test, test_pred, average="macro")]

model_results["LightGBM"] = {"validation": val_scores, "test": test_scores}

print("LIGHTGBM")
print(pd.DataFrame({"Validation": val_scores, "Test": test_scores}, index=metric_names).round(4))

LIGHTGBM
                   Validation    Test
Accuracy               0.9705  0.9695
Precision (macro)      0.9708  0.9701
Recall (macro)         0.9677  0.9660
F1 (macro)             0.9692  0.9679


In [19]:
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

catboost_model = CatBoostClassifier(iterations=400, depth=6, learning_rate=0.08,
                                    random_state=42, verbose=False)
catboost_model.fit(X_train, y_train)

val_pred = catboost_model.predict(X_val).ravel()
test_pred = catboost_model.predict(X_test).ravel()

val_scores = [accuracy_score(y_val, val_pred),
              precision_score(y_val, val_pred, average="macro"),
              recall_score(y_val, val_pred, average="macro"),
              f1_score(y_val, val_pred, average="macro")]
test_scores = [accuracy_score(y_test, test_pred),
               precision_score(y_test, test_pred, average="macro"),
               recall_score(y_test, test_pred, average="macro"),
               f1_score(y_test, test_pred, average="macro")]

model_results["CatBoost"] = {"validation": val_scores, "test": test_scores}

print("CATBOOST")
print(pd.DataFrame({"Validation": val_scores, "Test": test_scores}, index=metric_names).round(4))

CATBOOST
                   Validation    Test
Accuracy               0.9776  0.9743
Precision (macro)      0.9777  0.9732
Recall (macro)         0.9756  0.9712
F1 (macro)             0.9766  0.9722


In [20]:
catboost_scores = model_results["CatBoost"]

print("CATBOOST RESULTS")
print(pd.DataFrame({"Validation": catboost_scores["validation"],
                    "Test": catboost_scores["test"]},
                   index=metric_names).round(4))

CATBOOST RESULTS
                   Validation    Test
Accuracy               0.9776  0.9743
Precision (macro)      0.9777  0.9732
Recall (macro)         0.9756  0.9712
F1 (macro)             0.9766  0.9722


In [21]:
rows = []
for name, r in model_results.items():
    rows.append([name] + r["validation"] + r["test"])

columns = ["Model", "Val Acc", "Val Prec", "Val Rec", "Val F1",
           "Test Acc", "Test Prec", "Test Rec", "Test F1"]
comparison_df = pd.DataFrame(rows, columns=columns)
comparison_df = comparison_df.sort_values("Val F1", ascending=False).reset_index(drop=True)

print("MODEL COMPARISON (sorted by validation macro F1)")
print(comparison_df.round(4).to_string(index=False))

best_row = comparison_df.loc[0]
print("\nBest model by validation macro F1:", best_row["Model"])
print("\nTest results for", best_row["Model"], "(reported separately):")
print("  Accuracy :", round(best_row["Test Acc"], 4))
print("  Precision:", round(best_row["Test Prec"], 4))
print("  Recall   :", round(best_row["Test Rec"], 4))
print("  F1 score :", round(best_row["Test F1"], 4))

MODEL COMPARISON (sorted by validation macro F1)
        Model  Val Acc  Val Prec  Val Rec  Val F1  Test Acc  Test Prec  Test Rec  Test F1
     CatBoost   0.9776    0.9777   0.9756  0.9766    0.9743     0.9732    0.9712   0.9722
     LightGBM   0.9705    0.9708   0.9677  0.9692    0.9695     0.9701    0.9660   0.9679
      XGBoost   0.9657    0.9664   0.9621  0.9642    0.9681     0.9685    0.9648   0.9665
Random Forest   0.9524    0.9544   0.9445  0.9491    0.9486     0.9485    0.9408   0.9444
  Naive Bayes   0.7425    0.7543   0.6728  0.6806    0.7357     0.7487    0.6575   0.6596

Best model by validation macro F1: CatBoost

Test results for CatBoost (reported separately):
  Accuracy : 0.9743
  Precision: 0.9732
  Recall   : 0.9712
  F1 score : 0.9722


## 4. Prediction, SHAP and LIME for the selected combination

In [22]:
raw_features = compute_features(selected_assets, inv)

schema = {"feature_cols": feature_cols, "cat_cols": cat_cols, "bool_cols": bool_cols}
row_aligned = prepare_inference_row(raw_features, schema)

print("Selected assets        :", selected_assets)
print("Raw features generated :", len(raw_features))
print("Model input shape      :", row_aligned.shape)
print("Same columns as training, same order :", list(row_aligned.columns) == feature_cols)
print("Same columns and order as CatBoost   :", list(row_aligned.columns) == list(catboost_model.feature_names_))
print("Missing values in the row            :", int(row_aligned.isna().sum().sum()))

Selected assets        : ['AST1066', 'AST1079', 'AST1082']
Raw features generated : 60
Model input shape      : (1, 71)
Same columns as training, same order : True
Same columns and order as CatBoost   : True
Missing values in the row            : 0


In [23]:
pred_index = int(np.array(catboost_model.predict(row_aligned)).ravel()[0])
predicted_label = le.inverse_transform([pred_index])[0]

probabilities = catboost_model.predict_proba(row_aligned)[0]
prob_table = dict(zip(le.classes_, probabilities))

print("Selected assets    :", selected_assets)
print("Seen in training   :", not inv.is_new_combination(selected_assets))
print("PREDICTED CLASS    :", predicted_label)
print("\nClass probabilities:")
for cls in ["Low", "Medium", "High"]:
    print(f"  {cls:<7}: {prob_table[cls]:.1%}")

Selected assets    : ['AST1066', 'AST1079', 'AST1082']
Seen in training   : False
PREDICTED CLASS    : Low

Class probabilities:
  Low    : 92.9%
  Medium : 7.1%
  High   : 0.0%


In [24]:
feature_table = pd.DataFrame({"feature": list(raw_features.keys()),
                              "value": list(raw_features.values())})
feature_table.index = range(1, len(feature_table) + 1)

print("Selected assets :", selected_assets)
print("Predicted class :", predicted_label, "(from CatBoost, not from the dataset label)")
print()
print(feature_table.to_string())

differs = [k for k, v in raw_features.items() if k in row_aligned.columns and row_aligned[k].iloc[0] != v]
print("\nFeatures whose value changed between raw and model input:", differs)

Selected assets : ['AST1066', 'AST1079', 'AST1082']
Predicted class : Low (from CatBoost, not from the dataset label)

                                 feature     value
1                       combination_size         3
2                   asset_type_diversity         1
3                   combined_asset_value      11.1
4       combined_information_sensitivity       Low
5      information_complementarity_score       0.0
6               shared_information_count         1
7              information_overlap_score      0.33
8                 data_linkability_score       0.0
9   information_reconstruction_potential       0.0
10           contextual_information_gain       0.0
11           confidentiality_requirement       Low
12                 integrity_requirement       Low
13              availability_requirement    Medium
14                    relationship_count         0
15                 relationship_strength       0.0
16                      dependency_count         0
17            

In [25]:
from catboost import Pool

class_index = list(le.classes_).index(predicted_label)

shap_all = catboost_model.get_feature_importance(Pool(row_aligned), type="ShapValues")
print("Shape of SHAP output (rows, classes, features + base):", shap_all.shape)

shap_for_class = shap_all[0, class_index, :-1]
base_value = shap_all[0, class_index, -1]
raw_score = catboost_model.predict(row_aligned, prediction_type="RawFormulaVal")[0][class_index]

print("SHAP explains the existing CatBoost prediction. It does not make a prediction.")
print("Class being explained :", predicted_label, "(position", class_index, "in", list(le.classes_), ")")
print("Base value + sum of SHAP contributions :", round(base_value + shap_for_class.sum(), 4))
print("CatBoost raw score for this class      :", round(raw_score, 4))

shap_table = pd.DataFrame({"feature": feature_cols,
                           "value": row_aligned.iloc[0].values,
                           "shap": shap_for_class})
shap_table["direction"] = np.where(shap_table["shap"] > 0, "toward " + predicted_label, "away from " + predicted_label)
top_shap = shap_table.reindex(shap_table["shap"].abs().sort_values(ascending=False).index).head(10)

print("\nTop 10 features by size of contribution:")
print(top_shap.round(4).to_string(index=False))

Shape of SHAP output (rows, classes, features + base): (1, 3, 72)
SHAP explains the existing CatBoost prediction. It does not make a prediction.
Class being explained : Low (position 1 in ['High', 'Low', 'Medium'] )
Base value + sum of SHAP contributions : 3.6983
CatBoost raw score for this class      : 3.6983

Top 10 features by size of contribution:
                             feature value    shap     direction
           emergent_capability_score   0.1  2.9748    toward Low
      combined_control_effectiveness  4.14 -1.0659 away from Low
               control_effectiveness  4.14 -0.8934 away from Low
       potential_reputational_impact  2.93 -0.7678 away from Low
            attack_feasibility_score  0.18  0.5231    toward Low
information_reconstruction_potential   0.0  0.4700    toward Low
            risk_amplification_score   0.1  0.3978    toward Low
          potential_financial_impact  1.33  0.3016    toward Low
         potential_regulatory_impact   1.0  0.2929    toward 

In [26]:
from lime.lime_tabular import LimeTabularExplainer

class_index = list(le.classes_).index(predicted_label)
background = X_train.sample(300, random_state=42)
dummy_positions = [feature_cols.index(c) for c in new_columns]

lime_explainer = LimeTabularExplainer(training_data=background.values.astype(float),
                                      feature_names=feature_cols,
                                      class_names=list(le.classes_),
                                      categorical_features=dummy_positions,
                                      mode="classification",
                                      random_state=42)

def model_probabilities(rows):
    return catboost_model.predict_proba(pd.DataFrame(rows, columns=feature_cols))

lime_result = lime_explainer.explain_instance(row_aligned.iloc[0].values.astype(float),
                                              model_probabilities,
                                              labels=(class_index,),
                                              num_features=10,
                                              num_samples=3000)

print("LIME explains the existing CatBoost prediction. It does not make a prediction.")
print("Class being explained       :", predicted_label)
print("CatBoost probability        :", round(prob_table[predicted_label], 4))
print("LIME local model estimate   :", round(lime_result.local_pred[0], 4))
print("LIME local fit quality (R2) :", round(lime_result.score, 4))

lime_table = pd.DataFrame(lime_result.as_list(label=class_index), columns=["local condition", "weight"])
lime_table["direction"] = np.where(lime_table["weight"] > 0, "toward " + predicted_label, "away from " + predicted_label)

print("\nTop 10 local conditions:")
print(lime_table.round(4).to_string(index=False))

LIME explains the existing CatBoost prediction. It does not make a prediction.
Class being explained       : Low
CatBoost probability        : 0.9293
LIME local model estimate   : 0.7133
LIME local fit quality (R2) : 0.7913

Top 10 local conditions:
                             local condition  weight     direction
           emergent_capability_score <= 0.10  0.7273    toward Low
      combined_control_effectiveness <= 4.41 -0.1116 away from Low
               control_effectiveness <= 4.41 -0.0913 away from Low
information_reconstruction_potential <= 0.00  0.0626    toward Low
                access_overlap_score <= 0.00  0.0610    toward Low
          potential_financial_impact <= 1.35  0.0567    toward Low
        potential_reputational_impact > 2.13 -0.0530 away from Low
            risk_amplification_score <= 0.15  0.0454    toward Low
                   data_flow_present <= 0.00 -0.0395 away from Low
                   shared_user_count <= 0.00  0.0365    toward Low


In [27]:
lime_features = []
for rule in lime_table["local condition"]:
    matches = [f for f in feature_cols if f in rule]
    lime_features.append(max(matches, key=len))
lime_table["feature"] = lime_features

shap_direction = dict(zip(top_shap["feature"], top_shap["direction"]))
lime_direction = dict(zip(lime_table["feature"], lime_table["direction"]))
both = [f for f in top_shap["feature"] if f in lime_direction]

print("Prediction explained :", predicted_label, "for", selected_assets)
print("Features in SHAP top 10 :", len(shap_direction))
print("Features in LIME top 10 :", len(lime_direction))
print("Features in both        :", len(both))

comparison = pd.DataFrame({"feature": both,
                           "SHAP direction": [shap_direction[f] for f in both],
                           "LIME direction": [lime_direction[f] for f in both]})
comparison["agree"] = comparison["SHAP direction"] == comparison["LIME direction"]

print("\nFeatures found by both methods:")
print(comparison.to_string(index=False))
print("\nOnly in SHAP top 10:", [f for f in shap_direction if f not in lime_direction])
print("Only in LIME top 10:", [f for f in lime_direction if f not in shap_direction])

Prediction explained : Low for ['AST1066', 'AST1079', 'AST1082']
Features in SHAP top 10 : 10
Features in LIME top 10 : 10
Features in both        : 7

Features found by both methods:
                             feature SHAP direction LIME direction  agree
           emergent_capability_score     toward Low     toward Low   True
      combined_control_effectiveness  away from Low  away from Low   True
               control_effectiveness  away from Low  away from Low   True
       potential_reputational_impact  away from Low  away from Low   True
information_reconstruction_potential     toward Low     toward Low   True
            risk_amplification_score     toward Low     toward Low   True
          potential_financial_impact     toward Low     toward Low   True

Only in SHAP top 10: ['attack_feasibility_score', 'potential_regulatory_impact', 'reputational_exposure']
Only in LIME top 10: ['access_overlap_score', 'data_flow_present', 'shared_user_count']


## 5. Candidate ISO mapping (how Tables 1 and 2 were built, kept for explanation)
Every mapping is a candidate until a human reviews it.

In [28]:
iso_rows = [
    ("A.5.9",  "Inventory of information and other associated assets", "Organizational", "Asset inventory"),
    ("A.5.12", "Classification of information",       "Organizational", "Information classification and protection"),
    ("A.5.13", "Labelling of information",            "Organizational", "Information classification and protection"),
    ("A.5.14", "Information transfer",                "Organizational", "Information classification and protection"),
    ("A.8.12", "Data leakage prevention",             "Technological",  "Information classification and protection"),
    ("A.5.15", "Access control",                      "Organizational", "Access and identity"),
    ("A.5.16", "Identity management",                 "Organizational", "Access and identity"),
    ("A.5.17", "Authentication information",          "Organizational", "Access and identity"),
    ("A.5.18", "Access rights",                       "Organizational", "Access and identity"),
    ("A.8.2",  "Privileged access rights",            "Technological",  "Access and identity"),
    ("A.8.3",  "Information access restriction",      "Technological",  "Access and identity"),
    ("A.8.5",  "Secure authentication",               "Technological",  "Access and identity"),
    ("A.8.8",  "Management of technical vulnerabilities", "Technological", "Vulnerability management"),
    ("A.8.9",  "Configuration management",            "Technological",  "Vulnerability management"),
    ("A.8.15", "Logging",                             "Technological",  "Logging and monitoring"),
    ("A.8.16", "Monitoring activities",               "Technological",  "Logging and monitoring"),
    ("A.8.24", "Use of cryptography",                 "Technological",  "Cryptography"),
]

iso_reference = pd.DataFrame(iso_rows, columns=["iso_control_id", "iso_control_title", "iso_theme", "topic_group"])
iso_reference["title_verified"] = "Checked against a secondary source; confirm against the ISO text"

print("ISO/IEC 27001:2022 Annex A - candidate control shortlist:", len(iso_reference), "controls")
print(iso_reference[["iso_control_id", "iso_control_title", "topic_group"]].to_string(index=False))
print()
print(iso_reference["topic_group"].value_counts().to_string())

ISO/IEC 27001:2022 Annex A - candidate control shortlist: 17 controls
iso_control_id                                    iso_control_title                               topic_group
         A.5.9 Inventory of information and other associated assets                           Asset inventory
        A.5.12                        Classification of information Information classification and protection
        A.5.13                             Labelling of information Information classification and protection
        A.5.14                                 Information transfer Information classification and protection
        A.8.12                              Data leakage prevention Information classification and protection
        A.5.15                                       Access control                       Access and identity
        A.5.16                                  Identity management                       Access and identity
        A.5.17                           Authentic

In [29]:
controls_df = inv.controls_df

category_by_name = controls_df.groupby("control_name")["control_category"].agg(lambda s: ", ".join(sorted(s.unique())))
record_counts = controls_df["control_name"].value_counts()

erg_control_to_iso = pd.DataFrame({"erg_control_name": record_counts.index,
                                   "erg_control_category": category_by_name[record_counts.index].values,
                                   "record_count": record_counts.values})

erg_control_to_iso["candidate_iso_ids"] = ""
erg_control_to_iso["fit"] = "Not mapped yet"
erg_control_to_iso["mapping_rationale"] = ""
erg_control_to_iso["reviewed_by_human"] = "No"

print("Distinct control names in 02e_Controls:", len(erg_control_to_iso))
print("Total control records                 :", int(erg_control_to_iso["record_count"].sum()))
print("Names with more than one category     :", int(erg_control_to_iso["erg_control_category"].str.contains(",").sum()))
print()
print(erg_control_to_iso[["erg_control_name", "erg_control_category", "record_count"]].to_string(index=False))

Distinct control names in 02e_Controls: 42
Total control records                 : 3229
Names with more than one category     : 0

                   erg_control_name erg_control_category  record_count
                   Patch Management           Preventive           464
EDR (Endpoint Detection & Response) Detective/Corrective           430
                                MFA           Preventive           280
     Mobile Device Management (MDM)           Preventive           260
                  Endpoint Firewall           Preventive           253
                     Access Control           Preventive           248
               Full Disk Encryption           Preventive           245
               Network Segmentation           Preventive           233
                   Automated Backup Detective/Corrective           212
          Firmware Patch Management           Preventive            99
                 Encryption at Rest           Preventive            53
                 

In [30]:
batch1 = {
    "Patch Management": ("A.8.8", "Direct",
        "Patching closes known weaknesses in software, which is the purpose of managing technical vulnerabilities."),
    "EDR (Endpoint Detection & Response)": ("A.8.16", "Partial",
        "EDR watches endpoints for suspicious activity, which relates to monitoring. Its main purpose is malware protection, which is not in our shortlist."),
    "MFA": ("A.8.5", "Direct",
        "Multi-factor authentication is a way of making sign-in secure."),
    "Mobile Device Management (MDM)": ("", "None",
        "MDM manages and secures mobile devices. The shortlist has no device-management control, so there is no defensible match."),
    "Endpoint Firewall": ("", "None",
        "A firewall limits network traffic on a device. Network security controls are not in the shortlist."),
    "Access Control": ("A.5.15; A.8.3", "Direct",
        "Access control decides who may use which assets (A.5.15) and restricts access to information (A.8.3)."),
    "Full Disk Encryption": ("A.8.24", "Direct",
        "Disk encryption is a use of cryptography to protect data stored on a device."),
    "Network Segmentation": ("", "None",
        "Segmentation separates networks into zones. Network separation is not in the shortlist."),
    "Automated Backup": ("", "None",
        "Backup protects availability and recovery. A backup control is not in the shortlist."),
    "Firmware Patch Management": ("A.8.8", "Direct",
        "Firmware updates fix vulnerabilities in device firmware, which is part of managing technical vulnerabilities."),
    "Encryption at Rest": ("A.8.24", "Direct",
        "Encrypting stored data is a use of cryptography."),
    "Administrative MFA": ("A.8.5; A.8.2", "Direct",
        "MFA on administrator accounts secures sign-in (A.8.5) and protects privileged access rights (A.8.2)."),
    "Configuration Backup": ("A.8.9", "Partial",
        "Keeping copies of configurations supports configuration management, but its main purpose is recovery (backup), which is not in the shortlist."),
    "Logging": ("A.8.15", "Direct",
        "Logging records events and activity, which is what the logging control covers."),
}

valid_ids = set(iso_reference["iso_control_id"])
for name, (ids, fit, why) in batch1.items():
    row = erg_control_to_iso["erg_control_name"] == name
    assert row.sum() == 1, name + " was not found exactly once in the table"
    for one_id in [i for i in ids.split("; ") if i]:
        assert one_id in valid_ids, one_id + " is not in iso_reference"
    erg_control_to_iso.loc[row, "candidate_iso_ids"] = ids
    erg_control_to_iso.loc[row, "fit"] = fit
    erg_control_to_iso.loc[row, "mapping_rationale"] = why

done = erg_control_to_iso[erg_control_to_iso["fit"] != "Not mapped yet"]
print("Rows mapped so far:", len(done), "of", len(erg_control_to_iso))
print(done["fit"].value_counts().to_string())
print()
print(done[["erg_control_name", "candidate_iso_ids", "fit"]].to_string(index=False))

Rows mapped so far: 14 of 42
fit
Direct     8
None       4
Partial    2

                   erg_control_name candidate_iso_ids     fit
                   Patch Management             A.8.8  Direct
EDR (Endpoint Detection & Response)            A.8.16 Partial
                                MFA             A.8.5  Direct
     Mobile Device Management (MDM)                      None
                  Endpoint Firewall                      None
                     Access Control     A.5.15; A.8.3  Direct
               Full Disk Encryption            A.8.24  Direct
               Network Segmentation                      None
                   Automated Backup                      None
          Firmware Patch Management             A.8.8  Direct
                 Encryption at Rest            A.8.24  Direct
                 Administrative MFA      A.8.5; A.8.2  Direct
               Configuration Backup             A.8.9 Partial
                            Logging            A.8.15  Dire

In [31]:
batch2 = {
    "Configuration Hardening": ("A.8.9", "Direct",
        "Hardening sets secure configurations on systems, which is what configuration management covers."),
    "Change Management": ("", "None",
        "Change management controls how changes are approved and made. A change-management control is not in the shortlist."),
    "Database Activity Logging": ("A.8.15", "Direct",
        "It records activity on databases, which is what the logging control covers."),
    "Access Control List": ("A.5.15; A.8.3", "Direct",
        "An access control list states who may use a resource, applying access rules (A.5.15) and restricting access to information (A.8.3)."),
    "File Access Logging": ("A.8.15", "Direct",
        "It records who accessed files, which is a form of logging."),
    "Database Access Control": ("A.5.15; A.8.3", "Direct",
        "It limits who can reach database information, matching access control (A.5.15) and information access restriction (A.8.3)."),
    "Secrets Management": ("A.5.17; A.8.24", "Direct",
        "Secrets such as passwords and keys are authentication information, and managing them is the purpose of A.5.17. Where the secrets are cryptographic keys, A.8.24 also relates."),
    "Bucket/Access Policy Enforcement": ("A.5.15; A.8.3", "Direct",
        "Storage access policies decide who can read the stored data, matching access control and information access restriction."),
    "Access Logging": ("A.8.15", "Direct",
        "It records access events, which is a form of logging."),
    "MFA for Administrative Access": ("A.8.5; A.8.2", "Direct",
        "MFA on administrator access secures sign-in (A.8.5) and protects privileged access rights (A.8.2)."),
    "Offsite/Immutable Replication": ("", "None",
        "It keeps protected copies of data away from the main site for recovery. Backup and redundancy are not in the shortlist."),
    "Retention Policy Enforcement": ("", "None",
        "It keeps or deletes information after a set period. A retention or deletion control is not in the shortlist."),
    "MFA / SSO": ("A.8.5; A.5.16", "Direct",
        "MFA secures sign-in (A.8.5). Single sign-on relies on one managed identity per user, which relates to identity management (A.5.16)."),
    "Application Logging": ("A.8.15", "Direct",
        "It records events inside applications, which is a form of logging."),
}

valid_ids = set(iso_reference["iso_control_id"])
for name, (ids, fit, why) in batch2.items():
    row = erg_control_to_iso["erg_control_name"] == name
    assert row.sum() == 1, name + " was not found exactly once in the table"
    for one_id in [i for i in ids.split("; ") if i]:
        assert one_id in valid_ids, one_id + " is not in iso_reference"
    erg_control_to_iso.loc[row, "candidate_iso_ids"] = ids
    erg_control_to_iso.loc[row, "fit"] = fit
    erg_control_to_iso.loc[row, "mapping_rationale"] = why

done = erg_control_to_iso[erg_control_to_iso["fit"] != "Not mapped yet"]
print("Rows mapped so far:", len(done), "of", len(erg_control_to_iso))
print(done["fit"].value_counts().to_string())
print()
print(done[["erg_control_name", "candidate_iso_ids", "fit"]].tail(14).to_string(index=False))

Rows mapped so far: 28 of 42
fit
Direct     19
None        7
Partial     2

                erg_control_name candidate_iso_ids    fit
         Configuration Hardening             A.8.9 Direct
               Change Management                     None
       Database Activity Logging            A.8.15 Direct
             Access Control List     A.5.15; A.8.3 Direct
             File Access Logging            A.8.15 Direct
         Database Access Control     A.5.15; A.8.3 Direct
              Secrets Management    A.5.17; A.8.24 Direct
Bucket/Access Policy Enforcement     A.5.15; A.8.3 Direct
                  Access Logging            A.8.15 Direct
   MFA for Administrative Access      A.8.5; A.8.2 Direct
   Offsite/Immutable Replication                     None
    Retention Policy Enforcement                     None
                       MFA / SSO     A.8.5; A.5.16 Direct
             Application Logging            A.8.15 Direct


In [32]:
batch3 = {
    "Backup Integrity Verification": ("", "None",
        "It checks that backups can be restored. Backup is not in the shortlist."),
    "Role-Based Access Control": ("A.5.15; A.5.18", "Direct",
        "Roles decide who gets which access, matching access control (A.5.15) and access rights (A.5.18)."),
    "Third-Party Vendor Risk Review": ("", "None",
        "It reviews the risk of suppliers. Supplier relationships are not in the shortlist."),
    "Data Loss Prevention (DLP)": ("A.8.12", "Direct",
        "DLP stops sensitive data from leaving the organization, which is data leakage prevention."),
    "Activity Logging": ("A.8.15", "Direct",
        "It records user and system activity, which is a form of logging."),
    "Conditional Access Policies": ("A.5.15; A.8.5", "Partial",
        "These policies set conditions for sign-in, which relates to access control (A.5.15) and secure authentication (A.8.5). They are a partial match because the conditions vary by product."),
    "Logging and Monitoring": ("A.8.15; A.8.16", "Direct",
        "It covers both recording events (A.8.15) and watching them for unusual behaviour (A.8.16)."),
    "Log Integrity Protection": ("A.8.15", "Partial",
        "It protects logs from tampering. Logging (A.8.15) is the nearest topic in the shortlist, so this is a partial match."),
    "Retention Policy": ("", "None",
        "It sets how long information is kept. A retention or deletion control is not in the shortlist."),
    "Rate Limiting": ("", "None",
        "It limits how many requests a user or system can make. No defensible match in the shortlist."),
    "Web Application Firewall (WAF)": ("", "None",
        "It filters web traffic to an application. Network and web security are not in the shortlist."),
    "Privileged Access Management (PAM)": ("A.8.2", "Direct",
        "PAM manages and controls administrator-level accounts, which is privileged access rights."),
    "TLS/Encryption in Transit": ("A.8.24; A.5.14", "Direct",
        "Encrypting data in transit is use of cryptography (A.8.24) and protects information while it is transferred (A.5.14)."),
    "Periodic Access Review": ("A.5.18", "Direct",
        "Reviewing who has access and removing what is no longer needed is the management of access rights."),
}

valid_ids = set(iso_reference["iso_control_id"])
for name, (ids, fit, why) in batch3.items():
    row = erg_control_to_iso["erg_control_name"] == name
    assert row.sum() == 1, name + " was not found exactly once in the table"
    for one_id in [i for i in ids.split("; ") if i]:
        assert one_id in valid_ids, one_id + " is not in iso_reference"
    erg_control_to_iso.loc[row, "candidate_iso_ids"] = ids
    erg_control_to_iso.loc[row, "fit"] = fit
    erg_control_to_iso.loc[row, "mapping_rationale"] = why

print("Rows in table:", len(erg_control_to_iso))
print("Still 'Not mapped yet':", (erg_control_to_iso["fit"] == "Not mapped yet").sum())
print("Reviewed by human:", (erg_control_to_iso["reviewed_by_human"] == "Yes").sum())
print()
print(erg_control_to_iso["fit"].value_counts().to_string())
print()
print(erg_control_to_iso[["erg_control_name", "candidate_iso_ids", "fit"]].tail(14).to_string(index=False))

Rows in table: 42
Still 'Not mapped yet': 0
Reviewed by human: 0

fit
Direct     26
None       12
Partial     4

                  erg_control_name candidate_iso_ids     fit
     Backup Integrity Verification                      None
         Role-Based Access Control    A.5.15; A.5.18  Direct
    Third-Party Vendor Risk Review                      None
        Data Loss Prevention (DLP)            A.8.12  Direct
                  Activity Logging            A.8.15  Direct
       Conditional Access Policies     A.5.15; A.8.5 Partial
            Logging and Monitoring    A.8.15; A.8.16  Direct
          Log Integrity Protection            A.8.15 Partial
                  Retention Policy                      None
                     Rate Limiting                      None
    Web Application Firewall (WAF)                      None
Privileged Access Management (PAM)             A.8.2  Direct
         TLS/Encryption in Transit    A.8.24; A.5.14  Direct
            Periodic Access Revie

## 6. Compliance module checks (`compliance_mapper.py`)

In [33]:
import importlib
import compliance_mapper as cm
importlib.reload(cm)
print("module loaded, ISO rows:", len(cm.get_iso_reference()))
print("ERG controls mapped:", len(cm.get_erg_control_to_iso()))

module loaded, ISO rows: 17
ERG controls mapped: 42


In [34]:
from_module = cm.get_erg_control_to_iso()

names_in_sheet = set(inv.controls_df["control_name"])
print("Names match the sheet:", set(from_module["erg_control_name"]) == names_in_sheet)

nb = erg_control_to_iso.set_index("erg_control_name")[["candidate_iso_ids", "fit"]]
mod = from_module.set_index("erg_control_name")[["candidate_iso_ids", "fit"]]
print("IDs and fit match notebook table:", nb.sort_index().equals(mod.sort_index()))
print(from_module["fit"].value_counts().to_string())

Names match the sheet: True
IDs and fit match notebook table: True
fit
Direct     26
None       12
Partial     4


In [35]:
import importlib
import compliance_mapper as cm
importlib.reload(cm)

t3 = cm.get_risk_factor_to_control()
print("Table 3 rows:", len(t3), "| distinct features:", t3["risk_factor"].nunique())
print(t3["fit"].value_counts().to_string())

# Every name used in Part 3 must be a real model column or a real raw feature
known = set(raw_features.keys()) | set(feature_cols)
used = (set(t3["risk_factor"]) | set(cm.ACTIONABLE_VIA_CONTROL_RECORDS) | set(cm.DESCRIPTIVE_FACTORS))
print("Names not found in your features:", sorted(used - known))
print("Raw features not classified:", sorted(set(raw_features.keys()) - used))
print()
print(t3[["mapping_id", "risk_factor", "iso_control_id", "fit"]].to_string(index=False))

Table 3 rows: 19 | distinct features: 16
fit
Direct     13
Partial     6
Names not found in your features: []
Raw features not classified: []

mapping_id                     risk_factor iso_control_id     fit
     RF-01                    mfa_coverage          A.8.5  Direct
     RF-02             encryption_coverage         A.8.24  Direct
     RF-03             monitoring_coverage         A.8.15  Direct
     RF-04             monitoring_coverage         A.8.16 Partial
     RF-05            monitoring_gap_score         A.8.15  Direct
     RF-06            monitoring_gap_score         A.8.16 Partial
     RF-07         known_exploited_present          A.8.8  Direct
     RF-08          vulnerability_severity          A.8.8  Direct
     RF-09      common_vulnerability_count          A.8.8  Direct
     RF-10 vulnerability_interaction_score          A.8.8  Direct
     RF-11           common_weakness_score          A.8.8  Direct
     RF-12       privileged_access_present          A.8.2  Direct

## 7. `map_to_iso()` for the selected combination and the recommendation report

In [36]:
import importlib
import compliance_mapper as cm
importlib.reload(cm)

probability_of_prediction = float(catboost_model.predict_proba(row_aligned)[0][class_index])

report = cm.map_to_iso(
    asset_ids=selected_assets,
    predicted_label=predicted_label,
    probability=probability_of_prediction,
    shap_table=shap_table,
    lime_table=lime_table,
    inv=inv,
    feature_values=raw_features,
    feature_names=feature_cols,
)
cm.show_report(report)

The prediction is Low, so no recommendations are produced for this combination.
All items are candidates for human review. A predicted High or Medium combination does not prove ISO non-compliance.

CANDIDATE ISO CONTROLS
(none)

CONTROL-GAP RECORDS (ISO candidates from Table 2)
(none)

EVIDENCE FROM THE DATA
(none)

NOT MAPPED (one line per factor)
(none)


In [37]:
import pandas as pd

rows = []
rec = report["recommendations"]
ev = report["asset_evidence"]

# A. one row per (risk factor, candidate ISO control) from SHAP/LIME
for _, r in rec.iterrows():
    found = []
    for _, e in ev.iterrows():
        if r["risk_factor"] in str(e["risk_factor"]).split(", "):
            found.append(str(e["scope"]) + ": " + e["observation"])
    rows.append({
        "source": "Risk factor (" + r["explained_by"] + ")",
        "item": r["risk_factor"],
        "candidate_iso_control": r["iso_control_id"] + " " + r["iso_control_title"],
        "fit": r["fit"],
        "proposed_action": r["proposed_action"],
        "evidence_required": r["evidence_required"],
        "evidence_found_in_data": "; ".join(found) if found else "(none found)",
    })

# B. one row per control-gap record of the selected assets
for _, g in report["control_record_candidates"].iterrows():
    rows.append({
        "source": "Control record (02e_Controls)",
        "item": g["asset_id"] + " - " + g["erg_control_name"] + " (" + g["implementation_status"]
                + ", effectiveness " + str(g["control_effectiveness"]) + ")",
        "candidate_iso_control": g["candidate_iso_ids"],
        "fit": g["fit"],
        "proposed_action": "Review why this control is rated Low effectiveness on this asset and whether it applies.",
        "evidence_required": "Implementation and effectiveness record for this control on this asset.",
        "evidence_found_in_data": "Dataset records implementation status: " + g["implementation_status"],
    })

recommendation_report = pd.DataFrame(rows, columns=[
    "source", "item", "candidate_iso_control", "fit",
    "proposed_action", "evidence_required", "evidence_found_in_data"])
recommendation_report.insert(0, "no", range(1, len(recommendation_report) + 1))
recommendation_report["status"] = "Candidate - needs human review"
recommendation_report["reviewer_decision"] = "Pending"
recommendation_report["reviewer_notes"] = ""

print("Combination:", ", ".join(report["asset_ids"]), "| predicted:", report["predicted_label"])
print("Rows in report:", len(recommendation_report))
print(recommendation_report[["no", "source", "item", "candidate_iso_control", "fit", "reviewer_decision"]]
      .to_string(index=False))

Combination: AST1066, AST1079, AST1082 | predicted: Low
Rows in report: 0
Empty DataFrame
Columns: [no, source, item, candidate_iso_control, fit, reviewer_decision]
Index: []


## 8. Maximum Coverage (K = 10) and the final action plan
Covered means touched by a selected action. It does not mean compliant.

In [38]:
import importlib
import compliance_mapper as cm
importlib.reload(cm)

combinations_df = pd.read_excel(PATH, sheet_name="04_Generated_Combinations")
cov = cm.build_coverage_inputs(inv, combinations_df, labels_df)

print("Flagged (Medium/High) combinations:", cov["n_flagged"])
print("  with at least one mappable gap pair:", cov["n_flagged_with_pair"])
print("  with none (cannot be covered):", cov["n_flagged"] - cov["n_flagged_with_pair"])
print("Control-gap records:", cov["n_gap_records"], "| excluded (control has no ISO mapping):", cov["n_gap_records_unmapped"])
print("Candidate actions:", len(cov["actions"]), "| universe pairs to cover:", len(cov["universe"]))
print()
print(cov["actions"].sort_values("pairs_covered", ascending=False).head(5).to_string(index=False))

Flagged (Medium/High) combinations: 8295
  with at least one mappable gap pair: 6885
  with none (cannot be covered): 1410
Control-gap records: 1172 | excluded (control has no ISO mapping): 413
Candidate actions: 759 | universe pairs to cover: 12821

action_id asset_id                   erg_control_name implementation_status       iso_ids    fit  pairs_covered
CTRL01936  AST0648 Privileged Access Management (PAM)       Not Implemented         A.8.2 Direct            565
CTRL02369  AST0797                     Access Control           Implemented A.5.15; A.8.3 Direct            208
CTRL01814  AST0608                Access Control List Partially Implemented A.5.15; A.8.3 Direct            194
CTRL01839  AST0618                     Access Control       Not Implemented A.5.15; A.8.3 Direct            178
CTRL01785  AST0598                 Encryption at Rest Partially Implemented        A.8.24 Direct            167


In [39]:
importlib.reload(cm)
K = 10
selection = cm.greedy_max_coverage(cov, K)

print(selection[["step", "action_id", "asset_id", "erg_control_name", "iso_ids",
                 "new_pairs", "total_pairs_covered", "pct_of_universe"]].to_string(index=False))

last = selection.iloc[-1]
n_high = sum(1 for v in cov["label_of"].values() if v == "High")
n_med = cov["n_flagged"] - n_high
print()
print("With", len(selection), "actions:", int(last["total_pairs_covered"]), "of", len(cov["universe"]),
      "pairs covered (" + str(last["pct_of_universe"]) + "%)")
print("Flagged combinations touched:", int(last["combinations_touched"]), "of", cov["n_flagged"],
      "| High:", int(last["high_touched"]), "of", n_high,
      "| Medium:", int(last["medium_touched"]), "of", n_med)

print()
for k in (1, 3, 5, 10, 20, 50):
    s = cm.greedy_max_coverage(cov, k)
    print("K =", k, "->", s["pct_of_universe"].iloc[-1], "% of pairs")

 step action_id asset_id                    erg_control_name       iso_ids  new_pairs  total_pairs_covered  pct_of_universe
    1 CTRL01936  AST0648  Privileged Access Management (PAM)         A.8.2        565                  565              4.4
    2 CTRL02369  AST0797                      Access Control A.5.15; A.8.3        208                  773              6.0
    3 CTRL01814  AST0608                 Access Control List A.5.15; A.8.3        192                  965              7.5
    4 CTRL01785  AST0598                  Encryption at Rest        A.8.24        167                 1132              8.8
    5 CTRL01789  AST0599                 File Access Logging        A.8.15        163                 1295             10.1
    6 CTRL01839  AST0618                      Access Control A.5.15; A.8.3        162                 1457             11.4
    7 CTRL01096  AST0368                                 MFA         A.8.5        148                 1605             12.5
    8 CT

In [40]:
iso_title = cm.get_iso_reference().set_index("iso_control_id")["iso_control_title"].to_dict()
why_by_control = cm.get_erg_control_to_iso().set_index("erg_control_name")["mapping_rationale"].to_dict()

plan = selection[["step", "action_id", "asset_id", "erg_control_name", "implementation_status",
                  "iso_ids", "fit", "new_pairs", "total_pairs_covered", "pct_of_universe"]].copy()

plan["candidate_iso_controls"] = plan["iso_ids"].apply(
    lambda ids: "; ".join(i + " " + iso_title[i] for i in ids.split("; ")))
plan["why_this_iso_control"] = plan["erg_control_name"].map(why_by_control)
plan["proposed_action"] = ("Review the " + plan["erg_control_name"] + " control on " + plan["asset_id"]
                           + " (recorded as " + plan["implementation_status"]
                           + ", Low effectiveness) and decide whether to improve or confirm it.")
plan["evidence_to_verify"] = ("Implementation and effectiveness record for " + plan["erg_control_name"]
                              + " on " + plan["asset_id"] + "; confirm the candidate ISO control text applies.")
plan["status"] = "Candidate - needs human review"
plan["reviewer_decision"] = "Pending"
plan["reviewer_notes"] = ""

action_plan = plan[["step", "action_id", "asset_id", "erg_control_name", "implementation_status",
                    "candidate_iso_controls", "fit", "why_this_iso_control", "new_pairs",
                    "total_pairs_covered", "pct_of_universe", "proposed_action",
                    "evidence_to_verify", "status", "reviewer_decision", "reviewer_notes"]]

print("Action plan rows:", len(action_plan), "| K =", K)
print(action_plan[["step", "asset_id", "erg_control_name", "candidate_iso_controls",
                   "fit", "new_pairs", "reviewer_decision"]].to_string(index=False))
print()
print("First row in full:")
for col, val in action_plan.iloc[0].items():
    print(" ", col, ":", val)

Action plan rows: 10 | K = 10
 step asset_id                    erg_control_name                                      candidate_iso_controls     fit  new_pairs reviewer_decision
    1  AST0648  Privileged Access Management (PAM)                              A.8.2 Privileged access rights  Direct        565           Pending
    2  AST0797                      Access Control A.5.15 Access control; A.8.3 Information access restriction  Direct        208           Pending
    3  AST0608                 Access Control List A.5.15 Access control; A.8.3 Information access restriction  Direct        192           Pending
    4  AST0598                  Encryption at Rest                                  A.8.24 Use of cryptography  Direct        167           Pending
    5  AST0599                 File Access Logging                                              A.8.15 Logging  Direct        163           Pending
    6  AST0618                      Access Control A.5.15 Access control; A.8.3 In

In [41]:
import os, re, importlib, data_loader

code_folder = os.path.dirname(data_loader.__file__)
path = os.path.join(code_folder, "compliance_mapper.py")
print("File exists:", os.path.exists(path))
print("Size (bytes):", os.path.getsize(path) if os.path.exists(path) else "n/a")

import compliance_mapper as cm
importlib.reload(cm)
print("Imported from:", cm.__file__)

expected = ["get_iso_reference", "get_erg_control_to_iso", "get_risk_factor_to_control",
            "map_to_iso", "show_report", "build_coverage_inputs", "greedy_max_coverage",
            "_to_number", "_shows_gap", "_find_feature", "_pairs", "_asset_evidence"]
print("Missing functions:", [n for n in expected if not callable(getattr(cm, n, None))])

text = open(path, encoding="utf-8").read()
names = re.findall(r"^def (\w+)", text, re.M)
print("Functions in file:", len(names), "| defined twice:", sorted({n for n in names if names.count(n) > 1}))
print("Patch applied (WHAT TO CHECK in show_report):", "WHAT TO CHECK" in text)
print("Descriptive-prefix fallback present:", 'feat.startswith(b + "_")' in text)

print("ISO rows:", len(cm.get_iso_reference()),
      "| ERG controls:", len(cm.get_erg_control_to_iso()),
      "| Table 3 rows:", len(cm.get_risk_factor_to_control()))

File exists: True
Size (bytes): 42007
Imported from: C:\Users\Vidya Bodasakurthi\Downloads\GRC MINOR\Claude\compliance_mapper.py
Missing functions: []
Functions in file: 12 | defined twice: []
Patch applied (WHAT TO CHECK in show_report): True
Descriptive-prefix fallback present: True
ISO rows: 17 | ERG controls: 42 | Table 3 rows: 19


In [42]:
import pandas as pd
import data_loader as dl
import compliance_mapper as cm

users = pd.read_excel(PATH, sheet_name="02b_Users_and_Access")
vulns = pd.read_excel(PATH, sheet_name="02d_Vulnerabilities")
rels = pd.read_excel(PATH, sheet_name="02_Asset_Relationships")
combos = pd.read_excel(PATH, sheet_name="04_Generated_Combinations")
ctrls = inv.controls_df

def pairs_of(ids):
    return [(ids[i], ids[j]) for i in range(len(ids)) for j in range(i + 1, len(ids))]

def lacking(ids, names):
    has = set(ctrls[ctrls.asset_id.isin(ids) & ctrls.control_name.isin(names) &
                    ctrls.implementation_status.isin(["Implemented", "Partially Implemented"])].asset_id)
    return {a for a in ids if a not in has}

def users_of(a, privileged_only):
    d = users[users.asset_id == a]
    return set((d[d.privileged_access == True] if privileged_only else d).employee_id)

def flow(a1, a2):
    r = rels[((rels.source_asset_id == a1) & (rels.target_asset_id == a2)) |
             ((rels.source_asset_id == a2) & (rels.target_asset_id == a1))]
    return r.data_flow.notna().any() and (r.data_flow.astype(str) != "None").any()

# feature -> what the raw sheets say the evidence scopes should be
expected = {
    "mfa_coverage": lambda ids: lacking(ids, dl.MFA_NAMES),
    "encryption_coverage": lambda ids: lacking(ids, dl.ENCRYPTION_NAMES),
    "monitoring_coverage": lambda ids: lacking(ids, dl.MONITORING_NAMES),
    "monitoring_gap_score": lambda ids: lacking(ids, dl.MONITORING_NAMES),
    "known_exploited_present": lambda ids: {a for a in ids if (vulns[vulns.asset_id == a].known_exploited == True).any()},
    "privileged_access_present": lambda ids: {a + " and " + b for a, b in pairs_of(ids) if users_of(a, True) & users_of(b, True)},
    "shared_user_count": lambda ids: {a + " and " + b for a, b in pairs_of(ids) if users_of(a, False) & users_of(b, False)},
    "data_flow_present": lambda ids: {a + " and " + b for a, b in pairs_of(ids) if flow(a, b)},
}

all_ids = [[a.strip() for a in t.split(",")] for t in combos["asset_ids"]]
for feature, want_fn in expected.items():
    yes = next((ids for ids in all_ids if want_fn(ids)), None)       # a combination where it should fire
    no = next((ids for ids in all_ids if not want_fn(ids)), None)    # a combination where it should stay silent
    r1 = yes is not None and {s for s, _ in cm._asset_evidence(feature, yes, inv)} == want_fn(yes)
    r2 = no is not None and len(cm._asset_evidence(feature, no, inv)) == 0
    print(("PASS " if r1 and r2 else "FAIL ") + feature.ljust(26), "| fires on", yes, "| silent on", no)

# the counts written in the text must match the sheets
yes = next(ids for ids in all_ids if expected["privileged_access_present"](ids))
scope, text = cm._asset_evidence("privileged_access_present", yes, inv)[0]
a, b = scope.split(" and ")
print("Privileged:", text, "| sheet:", len(users_of(a, True) & users_of(b, True)))
yes = next(ids for ids in all_ids if expected["shared_user_count"](ids))
scope, text = cm._asset_evidence("shared_user_count", yes, inv)[0]
a, b = scope.split(" and ")
print("Shared users:", text, "| sheet:", len(users_of(a, False) & users_of(b, False)))

PASS mfa_coverage               | fires on ['AST0346', 'AST0618', 'AST0622'] | silent on ['AST0175', 'AST0527']
PASS encryption_coverage        | fires on ['AST0346', 'AST0618', 'AST0622'] | silent on ['AST0200', 'AST0309', 'AST0501']
PASS monitoring_coverage        | fires on ['AST0346', 'AST0618', 'AST0622'] | silent on ['AST0692', 'AST0728']
PASS monitoring_gap_score       | fires on ['AST0346', 'AST0618', 'AST0622'] | silent on ['AST0692', 'AST0728']
PASS known_exploited_present    | fires on ['AST0200', 'AST0309', 'AST0501'] | silent on ['AST0346', 'AST0618', 'AST0622']
PASS privileged_access_present  | fires on ['AST0165', 'AST0591', 'AST0614', 'AST0659'] | silent on ['AST0346', 'AST0618', 'AST0622']
PASS shared_user_count          | fires on ['AST0346', 'AST0618', 'AST0622'] | silent on ['AST1010', 'AST1055']
PASS data_flow_present          | fires on ['AST0290', 'AST0600', 'AST0767'] | silent on ['AST0346', 'AST0618', 'AST0622']
Privileged: 1 user(s) hold privileged access to b

In [43]:
import numpy as np
import pandas as pd
from catboost import Pool
import importlib
import compliance_mapper as cm
importlib.reload(cm)

def check(name, ok, detail=""):
    print(("PASS " if ok else "FAIL ") + name + ((" | " + str(detail)) if detail != "" else ""))

print("Selected assets:", selected_assets, "| predicted:", predicted_label)

# 1. the model, the encoded row and the column list agree
check("model feature names == feature_cols", list(catboost_model.feature_names_) == list(feature_cols), len(feature_cols))
check("row_aligned is 1 x 71 with the same columns", row_aligned.shape == (1, len(feature_cols)) and list(row_aligned.columns) == list(feature_cols), row_aligned.shape)

# 2. the predicted label is the model's own top class
proba = catboost_model.predict_proba(row_aligned)[0]
top_label = le.classes_[int(np.argmax(proba))]
check("predicted_label == model's top class", top_label == predicted_label, dict(zip(le.classes_, proba.round(3))))

# 3. shap_table is the model's SHAP for the predicted class (recomputed natively and compared)
ci = list(le.classes_).index(predicted_label)
fresh = catboost_model.get_feature_importance(Pool(row_aligned), type="ShapValues")[0, ci, :-1]
check("shap_table has columns feature, value, shap", {"feature", "value", "shap"} <= set(shap_table.columns), list(shap_table.columns))
aligned = shap_table.set_index("feature").loc[list(feature_cols), "shap"].values
check("shap_table equals a fresh CatBoost SHAP", np.allclose(aligned, fresh, atol=1e-6), "max diff " + format(np.abs(aligned - fresh).max(), ".2e"))

# 4. lime_table can be read by map_to_iso
check("lime_table has a rule column and a weight column", any("condition" in c or "rule" in c.lower() for c in lime_table.columns) and any("weight" in c.lower() for c in lime_table.columns), list(lime_table.columns))

# 5. run map_to_iso and trace every result back to SHAP / LIME / the data
p = float(proba[ci])
rep = cm.map_to_iso(selected_assets, predicted_label, p, shap_table, lime_table, inv,
                    feature_values=raw_features, feature_names=feature_cols)
print()
print(rep["summary"])

pos_shap = set(shap_table[shap_table["shap"] > 0].sort_values("shap", ascending=False).head(10)["feature"])
lime_feats = set()
for rule, w in zip(lime_table[[c for c in lime_table.columns if "condition" in c or "rule" in c.lower()][0]],
                   lime_table[[c for c in lime_table.columns if "weight" in c.lower()][0]]):
    if w > 0:
        hits = [f for f in feature_cols if f in str(rule)]
        lime_feats.add(max(hits, key=len) if hits else str(rule))
reported = set(rep["recommendations"]["risk_factor"]) | set(rep["not_mapped"]["risk_factor"])
check("every reported factor came from positive SHAP (top 10) or positive LIME", reported <= (pos_shap | lime_feats),
      sorted(reported - (pos_shap | lime_feats)))

if len(rep["recommendations"]):
    s = shap_table.set_index("feature")["shap"]
    bad = [r.risk_factor for r in rep["recommendations"].itertuples()
           if pd.notna(r.shap_value) and abs(r.shap_value - s[r.risk_factor]) > 1e-9]
    check("shap_value in each recommendation equals shap_table", not bad, bad)
    bad = [r.risk_factor for r in rep["recommendations"].itertuples()
           if r.risk_factor in raw_features and str(r.factor_value) != str(raw_features[r.risk_factor])
           and abs(float(r.factor_value) - float(raw_features[r.risk_factor])) > 1e-9]
    check("factor_value equals the generated feature value", not bad, bad)
    check("every iso_control_id is in the ISO reference", set(rep["recommendations"]["iso_control_id"]) <= set(cm.get_iso_reference()["iso_control_id"]))

raw_gaps = inv.controls_df[inv.controls_df.asset_id.isin(selected_assets) & (inv.controls_df.control_gap == True)]
listed = rep["control_record_candidates"]
check("control-gap records listed are real 02e gap rows",
      set(zip(listed.asset_id, listed.erg_control_name)) <= set(zip(raw_gaps.asset_id, raw_gaps.control_name)),
      "listed " + str(len(listed)) + " of " + str(len(raw_gaps)) + " real gap rows")
check("every item is marked as a candidate",
      all(str(x).startswith("Candidate") for x in rep["recommendations"].get("status", [])) and
      all(str(x).startswith("Candidate") for x in listed.get("status", [])))

Selected assets: ['AST1066', 'AST1079', 'AST1082'] | predicted: Low
PASS model feature names == feature_cols | 71
PASS row_aligned is 1 x 71 with the same columns | (1, 71)
PASS predicted_label == model's top class | {'High': np.float64(0.0), 'Low': np.float64(0.929), 'Medium': np.float64(0.071)}
PASS shap_table has columns feature, value, shap | ['feature', 'value', 'shap', 'direction']
PASS shap_table equals a fresh CatBoost SHAP | max diff 0.00e+00
PASS lime_table has a rule column and a weight column | ['local condition', 'weight', 'direction', 'feature']

The prediction is Low, so no recommendations are produced for this combination.
PASS every reported factor came from positive SHAP (top 10) or positive LIME | []
PASS control-gap records listed are real 02e gap rows | listed 0 of 3 real gap rows
PASS every item is marked as a candidate
